# CirCor PCG murmur detection: patient-level deep learning

This notebook builds a patient-level pipeline for the CirCor DigiScope phonocardiogram data. It uses log-Mel windows, a multi-scale CNN + BiLSTM + temporal attention model, class-balanced focal loss, recording and patient aggregation, and patient-level cross-validation. It reserves a final patient holdout before model development; that holdout is evaluated only after training and threshold selection are complete.

When prompted, choose a ZIP file containing the CirCor dataset folder. The notebook unpacks it and locates training_data.csv automatically. Standard Jupyter file pickers select files, so zip the dataset folder first if you have it as a folder.

The default task is binary murmur detection: records labeled Present and Absent are used, while Unknown/Inconclusive patients are excluded rather than silently treated as negative. Set TASK_MODE to three_class to retain Unknown as its own class. The CSV must contain patient-level murmur labels, and the audio files must be named like 12345_AV.wav (optional recording number before the location is supported).

The official CirCor release describes one CSV row per patient and audio files named by patient ID and auscultation site. Data versions differ in label spelling and layout, so this notebook resolves common column and label variants and prints the detected fields before training.

The model and Grad-CAM are research tools, not clinical decision systems. Report patient-level results with their split design; do not interpret them as diagnostic performance without external validation.


## 1. Install notebook dependencies

Run this cell once in the Jupyter environment. PyTorch should be installed for the local GPU/runtime; Colab usually includes it already. If a package is missing, restart the kernel after installation.


In [ ]:
%pip install -q librosa scikit-learn pandas matplotlib seaborn tqdm ipywidgets


## 2. Imports and configuration

The notebook will ask you to choose the dataset ZIP in the next step. The settings below control the model and training run.


In [ ]:
from pathlib import Path
import os, re, json, math, copy, hashlib, random, warnings, zipfile, shutil
from collections import Counter

import numpy as np
import pandas as pd
import librosa
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = torch.cuda.is_available()

# None auto-detects a common murmur column name. A column name can be supplied explicitly.
TARGET_COLUMN = None
TASK_MODE = "binary"   # "binary" drops Unknown/Inconclusive; "three_class" keeps it.

# Audio and feature settings
SAMPLE_RATE = 2000
N_FFT = 512
HOP_LENGTH = 128
N_MELS = 64
FMIN = 20
FMAX = 800
WINDOW_SECONDS = 6.0
N_WINDOWS = 3

# Training settings
N_FOLDS = 5
TEST_SIZE = 0.20
INNER_VAL_SIZE = 0.15
MAX_EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 7
FOCAL_GAMMA = 2.0
NUM_WORKERS = 0  # keep 0 for reliable Windows notebook execution
USE_AMP = torch.cuda.is_available()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RUN_DIR = Path.cwd() / "circor_run"
RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = Path.cwd() / "circor_logmel_cache.npz"
REBUILD_FEATURE_CACHE = False
UPLOAD_DIR = Path.cwd() / "circor_upload"
EXTRACT_DIR = UPLOAD_DIR / "extracted"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Run outputs:", RUN_DIR.resolve())


## 3. Choose and upload the dataset ZIP

Select the ZIP made from your CirCor dataset folder. In Google Colab, a file picker opens directly. In JupyterLab, use the upload widget; after selecting the ZIP and waiting for the upload to finish, run the next cell.


In [ ]:
try:
    from google.colab import files as colab_files
    IN_COLAB = True
    uploaded_files = colab_files.upload()
    if not uploaded_files:
        raise RuntimeError("No file was uploaded.")
    uploaded_name = next(iter(uploaded_files))
    archive_path = UPLOAD_DIR / Path(uploaded_name).name
    archive_path.write_bytes(uploaded_files[uploaded_name])
    print("Uploaded:", archive_path.name)
except ImportError:
    IN_COLAB = False
    import ipywidgets as widgets
    upload_widget = widgets.FileUpload(
        accept=".zip", multiple=False, description="Choose CirCor ZIP"
    )
    display(upload_widget)
    print("Choose a .zip file above. Wait for it to finish, then run the next cell.")


In [ ]:
if not IN_COLAB:
    uploaded_value = upload_widget.value
    if not uploaded_value:
        raise RuntimeError("Choose the CirCor ZIP in the upload widget above, then run this cell again.")
    if isinstance(uploaded_value, dict):  # ipywidgets 7
        uploaded_name, upload_entry = next(iter(uploaded_value.items()))
    else:  # ipywidgets 8
        upload_entry = uploaded_value[0]
        uploaded_name = upload_entry["name"]
    archive_path = UPLOAD_DIR / Path(uploaded_name).name
    archive_path.write_bytes(bytes(upload_entry["content"]))

if archive_path.suffix.lower() != ".zip":
    raise ValueError("Please upload a .zip file containing the CirCor dataset.")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(archive_path) as archive:
    root = EXTRACT_DIR.resolve()
    for member in archive.infolist():
        target = (EXTRACT_DIR / member.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError(f"Unsafe path in ZIP archive: {member.filename}")
    archive.extractall(EXTRACT_DIR)

csv_candidates = sorted(EXTRACT_DIR.rglob("training_data.csv"))
if not csv_candidates:
    raise FileNotFoundError(
        "No training_data.csv was found inside that ZIP. Please upload a ZIP that includes "
        "the CirCor patient label CSV and WAV recordings."
    )
CSV_PATH = csv_candidates[0]
DATA_ROOT = CSV_PATH.parent
print("Dataset extracted to:", EXTRACT_DIR.resolve())
print("Using metadata:", CSV_PATH)
print("Searching recordings under:", DATA_ROOT)


## 4. Read patient labels and match WAV recordings

The split unit is the patient ID. Every recording for a patient stays in one split. The loader accepts common PhysioNet column spellings and label encodings, but it stops if it cannot identify a patient ID or a murmur target.


In [ ]:
def normalize_column(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())

def normalize_patient_id(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    value = re.sub(r"\.0$", "", value)
    return value

def label_to_name(value):
    if pd.isna(value):
        return None
    s = str(value).strip().lower()
    s = re.sub(r"[_-]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    if s in {"present", "positive", "yes", "murmur", "1", "true"}:
        return "Present"
    if s in {"absent", "negative", "no", "no murmur", "2", "false"}:
        return "Absent"
    if s in {"unknown", "inconclusive", "unsure", "uncertain", "3"}:
        return "Unknown"
    return None

if not CSV_PATH.exists():
    raise FileNotFoundError(f"Could not find {CSV_PATH}. Edit DATA_ROOT or CSV_PATH above.")

metadata = pd.read_csv(CSV_PATH, dtype=str)
normalized = {normalize_column(c): c for c in metadata.columns}
patient_candidates = ["patientid", "patient", "patientidentifier"]
patient_col = next((normalized[k] for k in patient_candidates if k in normalized), None)
if patient_col is None:
    raise ValueError(f"Could not find a patient ID column. CSV columns: {list(metadata.columns)}")

if TARGET_COLUMN is not None:
    if TARGET_COLUMN not in metadata.columns:
        raise ValueError(f"TARGET_COLUMN={TARGET_COLUMN!r} is not in the CSV.")
    target_col = TARGET_COLUMN
else:
    target_candidates = ["murmur", "murmurs", "murmurlabel", "murmurstatus"]
    target_col = next((normalized[k] for k in target_candidates if k in normalized), None)
if target_col is None:
    raise ValueError(
        "Could not find a murmur label column. Set TARGET_COLUMN to the column containing "
        "Present/Absent/Unknown or 1/2/3 labels."
    )

metadata["patient_id"] = metadata[patient_col].map(normalize_patient_id)
metadata["label_name"] = metadata[target_col].map(label_to_name)
metadata = metadata.dropna(subset=["patient_id", "label_name"]).copy()

# Repeated CSV rows are allowed only if the patient-level label agrees.
label_counts = metadata.groupby("patient_id")["label_name"].nunique()
conflicts = label_counts[label_counts > 1]
if len(conflicts):
    raise ValueError(f"Conflicting labels for patient IDs: {conflicts.index.tolist()[:10]}")
patient_labels = metadata.drop_duplicates("patient_id").set_index("patient_id")["label_name"]

if TASK_MODE not in {"binary", "three_class"}:
    raise ValueError("TASK_MODE must be 'binary' or 'three_class'.")
if TASK_MODE == "binary":
    patient_labels = patient_labels[patient_labels.isin(["Absent", "Present"])]
    CLASSES = ["Absent", "Present"]
else:
    patient_labels = patient_labels[patient_labels.isin(["Absent", "Present", "Unknown"])]
    CLASSES = ["Absent", "Present", "Unknown"]

patient_labels = patient_labels[patient_labels.isin(CLASSES)].sort_index()
class_to_idx = {name: i for i, name in enumerate(CLASSES)}
patient_y = patient_labels.map(class_to_idx).astype(int)

audio_files = sorted(DATA_ROOT.rglob("*.wav"))
patient_id_set = set(patient_labels.index)
matched = []
for wav_path in audio_files:
    pid = normalize_patient_id(wav_path.stem.split("_", 1)[0])
    if pid in patient_id_set:
        matched.append({"patient_id": pid, "path": str(wav_path), "label": int(patient_y.loc[pid])})
recordings = pd.DataFrame(matched)
if recordings.empty:
    raise ValueError(
        f"No WAVs matched the patient IDs in {CSV_PATH}. Expected names such as 12345_AV.wav "
        "inside DATA_ROOT or one of its subfolders."
    )

patient_table = (
    recordings[["patient_id", "label"]].drop_duplicates("patient_id")
    .sort_values("patient_id").reset_index(drop=True)
)
print("Detected patient column:", patient_col, "| target column:", target_col)
print("Task mode:", TASK_MODE, "| classes:", CLASSES)
print("Patients:", len(patient_table), "| matched WAV recordings:", len(recordings))
print("Patient counts by class:")
print(patient_table["label"].map({v: k for k, v in class_to_idx.items()}).value_counts().reindex(CLASSES, fill_value=0))
print("Example recordings:")
display(recordings.head())


## 5. Extract or reload log-Mel windows

Each WAV becomes three fixed six-second windows, sampled at the beginning, middle, and end of the recording. Training randomly selects one of those windows each epoch; evaluation averages predictions over all three. Features are cached in the notebook's current folder and are rebuilt if their paths, sizes, or feature settings change.


In [ ]:
def feature_signature(frame):
    file_state = []
    for p in frame["path"]:
        stat = Path(p).stat()
        file_state.append((str(Path(p).resolve()), stat.st_size, stat.st_mtime_ns))
    payload = {
        "files": file_state,
        "sample_rate": SAMPLE_RATE, "n_fft": N_FFT, "hop": HOP_LENGTH,
        "n_mels": N_MELS, "fmin": FMIN, "fmax": FMAX,
        "window_seconds": WINDOW_SECONDS, "n_windows": N_WINDOWS,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

signature = feature_signature(recordings)
features = None
if CACHE_PATH.exists() and not REBUILD_FEATURE_CACHE:
    try:
        cached = np.load(CACHE_PATH, allow_pickle=False)
        if str(cached["signature"].item()) == signature:
            features = cached["features"]
            cached_paths = cached["record_paths"].astype(str).tolist()
            recordings = recordings.set_index("path").loc[cached_paths].reset_index()
            print("Loaded feature cache:", CACHE_PATH.resolve(), features.shape)
        else:
            print("Feature cache settings or audio files changed; rebuilding.")
    except Exception as exc:
        print("Could not use feature cache; rebuilding:", exc)

if features is None:
    frames_per_window = int(round(WINDOW_SECONDS * SAMPLE_RATE / HOP_LENGTH))
    feature_rows, valid_rows = [], []
    failed = []
    for row_idx, row in tqdm(recordings.iterrows(), total=len(recordings), desc="Log-Mel extraction"):
        try:
            audio, _ = librosa.load(row["path"], sr=SAMPLE_RATE, mono=True)
            audio = np.nan_to_num(audio.astype(np.float32))
            if len(audio) == 0:
                raise ValueError("empty audio")
            mel = librosa.feature.melspectrogram(
                y=audio, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH,
                n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0
            )
            db = librosa.power_to_db(mel, ref=np.max, top_db=80.0)
            spec = np.clip((db + 80.0) / 40.0 - 1.0, -1.0, 1.0).astype(np.float32)
            total_frames = spec.shape[1]
            if total_frames <= frames_per_window:
                starts = [0] * N_WINDOWS
            else:
                last_start = total_frames - frames_per_window
                starts = np.linspace(0, last_start, N_WINDOWS).round().astype(int).tolist()
            windows = []
            for start in starts:
                part = spec[:, start:start + frames_per_window]
                if part.shape[1] < frames_per_window:
                    part = np.pad(part, ((0, 0), (0, frames_per_window - part.shape[1])), constant_values=-1.0)
                windows.append(part[None, :, :])
            feature_rows.append(np.stack(windows, axis=0).astype(np.float16))
            valid_rows.append(row_idx)
        except Exception as exc:
            failed.append((row["path"], str(exc)))
    if failed:
        print(f"Skipped {len(failed)} unreadable recording(s). First few:", failed[:3])
    recordings = recordings.loc[valid_rows].reset_index(drop=True)
    features = np.stack(feature_rows).astype(np.float16)
    np.savez_compressed(
        CACHE_PATH, features=features, signature=np.array(signature),
        record_paths=recordings["path"].astype(str).to_numpy()
    )
    print("Saved feature cache:", CACHE_PATH.resolve(), features.shape)

assert len(recordings) == len(features)
patient_table = recordings[["patient_id", "label"]].drop_duplicates("patient_id").sort_values("patient_id").reset_index(drop=True)
print("Feature tensor shape:", features.shape, "(recording, window, channel, mel, time)")
print("Usable patients:", len(patient_table), "| usable recordings:", len(recordings))


## 6. Make the untouched patient holdout

The holdout is split once at patient level and is not used for checkpoint selection, early stopping, scheduler decisions, or threshold tuning. Five-fold evaluation runs only on the development patients.


In [ ]:
class_counts = patient_table["label"].value_counts().reindex(range(len(CLASSES)), fill_value=0)
if (class_counts < N_FOLDS + 1).any():
    readable = {CLASSES[i]: int(class_counts[i]) for i in range(len(CLASSES))}
    raise ValueError(
        f"Too few patients for a reliable {N_FOLDS}-fold run plus holdout: {readable}. "
        "Use TASK_MODE='binary' if Unknown is too sparse, or reduce N_FOLDS."
    )

dev_patients, test_patients = train_test_split(
    patient_table["patient_id"].to_numpy(),
    test_size=TEST_SIZE, random_state=SEED,
    stratify=patient_table["label"].to_numpy()
)
dev_patients, test_patients = set(dev_patients), set(test_patients)
print("Development patients:", len(dev_patients), "| untouched test patients:", len(test_patients))


## 7. Dataset, augmentation, and model

The network applies parallel small, medium, and dilated convolutions, then a bidirectional LSTM and learned temporal attention. SpecAugment masking, small log-Mel noise, modest time shifts, and mild low-frequency perturbations are applied to training windows only.


In [ ]:
class PCGWindowDataset(Dataset):
    def __init__(self, record_indices, augment=False):
        self.indices = np.asarray(record_indices, dtype=np.int64)
        self.augment = augment
        self.labels = recordings.iloc[self.indices]["label"].to_numpy(dtype=np.int64)
        patient_ids = recordings.iloc[self.indices]["patient_id"].to_numpy()
        counts = Counter(patient_ids)
        self.patient_weights = np.asarray([1.0 / counts[pid] for pid in patient_ids], dtype=np.float32)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, j):
        ridx = int(self.indices[j])
        widx = random.randrange(N_WINDOWS) if self.augment else 0
        x = torch.from_numpy(features[ridx, widx].astype(np.float32, copy=True))
        if self.augment:
            x = augment_logmel(x)
        return x, int(self.labels[j]), float(self.patient_weights[j])

def augment_logmel(x):
    # x: [1, mel, time], normalized log-Mel values
    if random.random() < 0.5:
        shift = random.randint(-4, 4)
        x = torch.roll(x, shifts=shift, dims=-1)
    if random.random() < 0.6:
        x = x + torch.randn_like(x) * random.uniform(0.005, 0.025)
    if random.random() < 0.4:
        # Mild low-frequency perturbation that does not erase the PCG structure.
        cutoff = random.randint(4, max(5, N_MELS // 8))
        x[:, :cutoff, :] *= random.uniform(0.85, 1.15)
    if random.random() < 0.7:
        width = random.randint(1, max(1, N_MELS // 8))
        start = random.randint(0, N_MELS - width)
        x[:, start:start + width, :] = -1.0
    if random.random() < 0.7:
        width = random.randint(1, max(1, x.shape[-1] // 8))
        start = random.randint(0, x.shape[-1] - width)
        x[:, :, start:start + width] = -1.0
    return x.clamp(-1.5, 1.5)

class MultiScaleBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        branch_ch = max(8, out_ch // 3)
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_ch, branch_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(branch_ch), nn.GELU()
        )
        self.branch5 = nn.Sequential(
            nn.Conv2d(in_ch, branch_ch, 5, padding=2, bias=False),
            nn.BatchNorm2d(branch_ch), nn.GELU()
        )
        self.branch_dilated = nn.Sequential(
            nn.Conv2d(in_ch, branch_ch, 3, padding=2, dilation=2, bias=False),
            nn.BatchNorm2d(branch_ch), nn.GELU()
        )
        self.project = nn.Sequential(
            nn.Conv2d(branch_ch * 3, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.MaxPool2d(2), nn.Dropout2d(0.12)
        )

    def forward(self, x):
        return self.project(torch.cat(
            [self.branch3(x), self.branch5(x), self.branch_dilated(x)], dim=1
        ))

class PCGMurmurNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.block1 = MultiScaleBlock(1, 48)
        self.block2 = MultiScaleBlock(48, 72)
        self.last_conv_map = None
        self.sequence = nn.LSTM(
            input_size=72, hidden_size=64, num_layers=1,
            batch_first=True, bidirectional=True
        )
        self.attention = nn.Sequential(nn.Linear(128, 64), nn.Tanh(), nn.Linear(64, 1))
        self.dropout = nn.Dropout(0.35)
        self.classifier = nn.Linear(128, n_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        self.last_conv_map = x
        # Average over Mel frequency; retain the temporal sequence for the BiLSTM.
        seq = x.mean(dim=2).transpose(1, 2)
        seq, _ = self.sequence(seq)
        weights = torch.softmax(self.attention(seq).squeeze(-1), dim=1)
        pooled = torch.sum(seq * weights.unsqueeze(-1), dim=1)
        return self.classifier(self.dropout(pooled))

print(PCGMurmurNet(len(CLASSES)))


## 8. Focal loss, patient aggregation, and metrics

Class weights come from patient counts in each training fold. Recording losses are additionally weighted so a patient with more WAV files does not automatically contribute more total training weight. For binary murmur detection, the Present threshold is chosen on validation predictions only.


In [ ]:
def focal_loss_per_sample(logits, targets, alpha, gamma=FOCAL_GAMMA):
    log_probs = torch.log_softmax(logits, dim=1)
    log_pt = log_probs.gather(1, targets[:, None]).squeeze(1)
    pt = log_pt.exp()
    alpha_t = alpha.gather(0, targets)
    return -alpha_t * (1.0 - pt).pow(gamma) * log_pt

def indices_for_patients(patient_ids):
    return np.flatnonzero(recordings["patient_id"].isin(set(patient_ids)).to_numpy())

@torch.no_grad()
def predict_patient_probabilities(model, patient_ids, batch_size=64):
    model.eval()
    record_indices = indices_for_patients(patient_ids)
    record_probabilities = np.zeros((len(record_indices), len(CLASSES)), dtype=np.float64)
    for start in range(0, len(record_indices), batch_size):
        batch_indices = record_indices[start:start + batch_size]
        batch = torch.from_numpy(features[batch_indices].astype(np.float32, copy=False))
        b, nwin, ch, mel, frames = batch.shape
        batch = batch.reshape(b * nwin, ch, mel, frames).to(DEVICE)
        logits = model(batch)
        probs = torch.softmax(logits, dim=1).reshape(b, nwin, -1).mean(dim=1)
        record_probabilities[start:start + len(batch_indices)] = probs.cpu().numpy()
    frame = recordings.iloc[record_indices][["patient_id", "label"]].copy().reset_index(drop=True)
    frame["row_in_batch"] = np.arange(len(frame))
    prob_cols = [f"p_{c}" for c in CLASSES]
    frame[prob_cols] = record_probabilities
    patient_rows = []
    for pid, group in frame.groupby("patient_id", sort=False):
        probs = group[prob_cols].to_numpy().mean(axis=0)
        patient_rows.append({"patient_id": pid, "label": int(group["label"].iloc[0]), **dict(zip(prob_cols, probs))})
    return pd.DataFrame(patient_rows)

def probabilities_from_frame(frame):
    return frame[[f"p_{c}" for c in CLASSES]].to_numpy(dtype=np.float64)

def tune_binary_threshold(frame):
    if len(CLASSES) != 2 or CLASSES[1] != "Present":
        return 0.5
    y = frame["label"].to_numpy(dtype=int)
    p = probabilities_from_frame(frame)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y, p, pos_label=1)
    if len(thresholds) == 0:
        return 0.5
    f1_values = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    return float(thresholds[int(np.nanargmax(f1_values))])

def metrics_for_frame(frame, threshold=None, show_report=True):
    y = frame["label"].to_numpy(dtype=int)
    probs = probabilities_from_frame(frame)
    if len(CLASSES) == 2 and threshold is not None:
        pred = (probs[:, 1] >= threshold).astype(int)
    else:
        pred = probs.argmax(axis=1)
    result = {
        "accuracy": accuracy_score(y, pred),
        "macro_precision": precision_score(y, pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y, pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
    }
    try:
        if len(CLASSES) == 2:
            result["roc_auc"] = roc_auc_score(y, probs[:, 1])
        else:
            result["roc_auc_ovr_macro"] = roc_auc_score(
                y, probs, labels=np.arange(len(CLASSES)), multi_class="ovr", average="macro"
            )
    except ValueError:
        result["roc_auc"] = np.nan
    if show_report:
        print(pd.Series(result).round(4))
        print(classification_report(y, pred, labels=np.arange(len(CLASSES)), target_names=CLASSES, zero_division=0))
        cm = confusion_matrix(y, pred, labels=np.arange(len(CLASSES)))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
        plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Patient-level confusion matrix")
        plt.show()
    return result, pred


## 9. Training helpers: early stopping and learning-rate scheduling

Within each CV fold, early stopping and ReduceLROnPlateau use a patient-level inner validation split. The outer fold is used only for that fold's evaluation. The final model's epoch count is selected from the five fold checkpoints, then it is trained on all development patients with a cosine learning-rate schedule.


In [ ]:
def class_alpha_from_patients(patient_ids):
    frame = patient_table[patient_table["patient_id"].isin(set(patient_ids))]
    counts = frame["label"].value_counts().reindex(range(len(CLASSES)), fill_value=0).to_numpy()
    if (counts == 0).any():
        raise ValueError(f"A training split is missing a class: {counts.tolist()}")
    weights = len(frame) / (len(CLASSES) * counts.astype(np.float64))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def patient_nll(frame):
    probs = np.clip(probabilities_from_frame(frame), 1e-8, 1.0)
    y = frame["label"].to_numpy(dtype=int)
    return float(-np.log(probs[np.arange(len(y)), y]).mean())

def fit_model(train_patient_ids, val_patient_ids=None, max_epochs=MAX_EPOCHS,
              run_name="model", fixed_epochs=False):
    train_patient_ids = set(train_patient_ids)
    train_indices = indices_for_patients(train_patient_ids)
    train_set = PCGWindowDataset(train_indices, augment=True)
    loader = DataLoader(
        train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=False,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
    )
    model = PCGMurmurNet(len(CLASSES)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    alpha = class_alpha_from_patients(train_patient_ids)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    if fixed_epochs or val_patient_ids is None:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
    else:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=2, min_lr=1e-6
        )

    best_state = None
    best_val = float("inf")
    best_epoch = max_epochs
    stale = 0
    history = []
    for epoch in range(1, max_epochs + 1):
        model.train()
        losses = []
        for xb, yb, wb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            wb = wb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(xb)
                per_sample = focal_loss_per_sample(logits, yb, alpha)
                loss = (per_sample * wb).sum() / wb.sum().clamp_min(1e-8)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))

        if val_patient_ids is not None and not fixed_epochs:
            val_frame = predict_patient_probabilities(model, val_patient_ids)
            val_loss = patient_nll(val_frame)
            scheduler.step(val_loss)
            history.append({"epoch": epoch, "train_focal": float(np.mean(losses)), "val_patient_nll": val_loss})
            print(f"{run_name} | epoch {epoch:02d} | train focal {np.mean(losses):.4f} | patient val NLL {val_loss:.4f}")
            if val_loss < best_val - 1e-4:
                best_val, best_epoch, stale = val_loss, epoch, 0
                best_state = copy.deepcopy(model.state_dict())
                torch.save(best_state, RUN_DIR / f"{run_name}_best.pt")
            else:
                stale += 1
                if stale >= PATIENCE:
                    print("Early stopping.")
                    break
        else:
            scheduler.step()
            history.append({"epoch": epoch, "train_focal": float(np.mean(losses))})
            print(f"{run_name} | epoch {epoch:02d}/{max_epochs} | train focal {np.mean(losses):.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    elif fixed_epochs or val_patient_ids is None:
        torch.save(model.state_dict(), RUN_DIR / f"{run_name}_final.pt")
    return model, best_epoch, pd.DataFrame(history)


## 10. Five-fold patient-level evaluation

Every patient appears in exactly one outer validation fold. Within each fold, a separate patient-level validation subset from the training patients controls early stopping and the binary threshold. Fold metrics are computed on patient-level averaged probabilities.


In [ ]:
dev_table = patient_table[patient_table["patient_id"].isin(dev_patients)].reset_index(drop=True)
if dev_table["label"].value_counts().min() < N_FOLDS:
    raise ValueError("The development split has fewer patients than folds in at least one class; reduce N_FOLDS or use binary mode.")

fold_splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_results = []
oof_parts = []
best_epochs = []

for fold, (train_pos, outer_val_pos) in enumerate(
    fold_splitter.split(dev_table["patient_id"], dev_table["label"]), start=1
):
    outer_train = dev_table.iloc[train_pos]
    outer_val = dev_table.iloc[outer_val_pos]
    min_count = int(outer_train["label"].value_counts().min())
    inner_fraction = min(0.30, max(INNER_VAL_SIZE, len(CLASSES) / max(min_count, 1)))
    inner_train_ids, inner_val_ids = train_test_split(
        outer_train["patient_id"].to_numpy(),
        test_size=inner_fraction, random_state=SEED + fold,
        stratify=outer_train["label"].to_numpy()
    )
    model, best_epoch, history = fit_model(
        inner_train_ids, inner_val_ids, max_epochs=MAX_EPOCHS, run_name=f"fold_{fold}"
    )
    best_epochs.append(best_epoch)

    inner_val_frame = predict_patient_probabilities(model, inner_val_ids)
    fold_threshold = tune_binary_threshold(inner_val_frame) if len(CLASSES) == 2 else None
    outer_frame = predict_patient_probabilities(model, outer_val["patient_id"].to_numpy())
    outer_frame["fold"] = fold
    oof_parts.append(outer_frame)
    fold_metrics, _ = metrics_for_frame(outer_frame, threshold=fold_threshold, show_report=False)
    fold_results.append({"fold": fold, "best_epoch": best_epoch, "threshold": fold_threshold, **fold_metrics})
    print(f"Fold {fold} | inner validation threshold: {fold_threshold}")
    print(pd.Series(fold_metrics).round(4))

cv_results = pd.DataFrame(fold_results)
display(cv_results)
print("\nCV mean and standard deviation:")
display(cv_results.drop(columns=["fold"]).agg(["mean", "std"]).round(4))
oof_frame = pd.concat(oof_parts, ignore_index=True)
assert oof_frame["patient_id"].nunique() == len(dev_patients), "Each development patient should have one OOF prediction."


## 11. Train the final model and select a validation-only threshold

The final threshold comes from development-set out-of-fold predictions, where each patient's prediction was produced by a model that did not train on that patient. The untouched test patients are not used to choose the threshold or epoch count.


In [ ]:
final_threshold = tune_binary_threshold(oof_frame) if len(CLASSES) == 2 else None
final_epochs = int(np.median(best_epochs))
final_epochs = max(3, final_epochs)
print("Final training epochs (median fold best epoch):", final_epochs)
print("Threshold selected using development OOF predictions:", final_threshold)

final_model, _, final_history = fit_model(
    dev_patients, val_patient_ids=None, max_epochs=final_epochs,
    run_name="final_model", fixed_epochs=True
)

model_path = RUN_DIR / "circor_patient_level_model.pt"
torch.save({
    "state_dict": final_model.state_dict(),
    "classes": CLASSES,
    "threshold": final_threshold,
    "settings": {
        "sample_rate": SAMPLE_RATE, "n_fft": N_FFT, "hop_length": HOP_LENGTH,
        "n_mels": N_MELS, "window_seconds": WINDOW_SECONDS, "n_windows": N_WINDOWS,
        "task_mode": TASK_MODE,
    },
}, model_path)
print("Saved final checkpoint:", model_path.resolve())


## 12. One final untouched test evaluation

This is the only cell that evaluates the holdout. Preserve these results as the final internal estimate; repeated changes based on the holdout would turn it into validation data.


In [ ]:
test_frame = predict_patient_probabilities(final_model, test_patients)
test_metrics, test_predictions = metrics_for_frame(
    test_frame, threshold=final_threshold, show_report=True
)
test_frame["predicted_label"] = [CLASSES[i] for i in test_predictions]
test_frame["actual_label"] = test_frame["label"].map({i: c for i, c in enumerate(CLASSES)})
test_frame.to_csv(RUN_DIR / "patient_level_test_predictions.csv", index=False)
pd.Series(test_metrics).to_csv(RUN_DIR / "patient_level_test_metrics.csv", header=["value"])
print("Saved test predictions and metrics in:", RUN_DIR.resolve())


## 13. Grad-CAM for a patient recording

This visualizes which time-frequency areas of one six-second window contributed to the selected model output. It is a model explanation, not evidence that a highlighted area is a clinically meaningful murmur.


In [ ]:
def plot_gradcam(record_index, window_index=0, target_class=None):
    final_model.eval()
    x = torch.from_numpy(features[record_index, window_index].astype(np.float32, copy=True))[None].to(DEVICE)
    final_model.zero_grad(set_to_none=True)
    logits = final_model(x)
    if target_class is None:
        target_class = int(logits.argmax(dim=1).item())
    conv_map = final_model.last_conv_map
    grad = torch.autograd.grad(logits[0, target_class], conv_map, retain_graph=False)[0]
    weights = grad.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((weights * conv_map).sum(dim=1, keepdim=True))
    cam = torch.nn.functional.interpolate(
        cam, size=x.shape[-2:], mode="bilinear", align_corners=False
    )[0, 0].detach().cpu().numpy()
    cam = cam / (cam.max() + 1e-8)
    spec = x[0, 0].detach().cpu().numpy()
    pid = recordings.iloc[record_index]["patient_id"]
    with torch.no_grad():
        predicted = CLASSES[int(torch.softmax(logits, dim=1).argmax(dim=1).item())]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
    axes[0].imshow(spec, origin="lower", aspect="auto", cmap="magma")
    axes[0].set_title(f"Log-Mel | patient {pid}")
    axes[1].imshow(spec, origin="lower", aspect="auto", cmap="gray")
    axes[1].imshow(cam, origin="lower", aspect="auto", cmap="jet", alpha=0.48)
    axes[1].set_title(f"Grad-CAM for {CLASSES[target_class]} | predicted {predicted}")
    for ax in axes:
        ax.set_xlabel("Time frames"); ax.set_ylabel("Mel bins")
    plt.show()

# Example: display the first held-out patient's first recording and first window.
example_patient = next(iter(test_patients))
example_record_index = int(recordings.index[recordings["patient_id"] == example_patient][0])
plot_gradcam(example_record_index, window_index=0)


## 14. Optional SHAP analysis

SHAP can be slow and memory-intensive for a CNN-RNN. Run this only after the main pipeline completes. The example explains one held-out window using a small background sample; the test set remains evaluation-only, and these explanations should not guide threshold or model selection.


In [ ]:
# Uncomment to install if needed:
# %pip install -q shap

RUN_SHAP = False
if RUN_SHAP:
    import shap
    background_indices = np.random.choice(len(recordings), size=min(8, len(recordings)), replace=False)
    background = torch.from_numpy(features[background_indices, 0].astype(np.float32, copy=True)).to(DEVICE)
    explained = torch.from_numpy(features[[example_record_index], 0].astype(np.float32, copy=True)).to(DEVICE)
    explainer = shap.GradientExplainer(final_model, background)
    shap_values = explainer.shap_values(explained)
    print("SHAP output type:", type(shap_values))
    print("For multi-class output, inspect the class-specific arrays and plot a time-frequency summary.")


## Notes on interpreting the run

- All partitioning and aggregation are patient-level; multiple auscultation sites from one patient remain together.
- Cross-validation scores are internal estimates from the development split. The one-time holdout gives a final internal estimate, not external validation.
- Binary mode removes Unknown/Inconclusive labels from both training and held-out evaluation. Use three_class mode if retaining that uncertainty category is important and there are enough patients for five folds.
- ROC-AUC uses patient-level averaged probabilities. The binary decision threshold is selected only from development validation predictions.
- Consider repeating the entire patient holdout with a pre-registered seed or an external cohort before making clinical or deployment claims.

Dataset reference: [CirCor DigiScope Phonocardiogram Dataset on PhysioNet](https://physionet.org/content/circor-heart-sound/1.0.0/).
